# Chapter 18: Proven, not demonstrated

The notebook version of [`README.md`](README.md) and [`run.py`](run.py), generated by `make notebooks`. Run the cells in order. The lesson stops to ask for your predictions; answer in the box under the cell.

In [ ]:
# Setup: tell LANTERN which chapter this is, and give the lesson code the __file__
# a script has (the kernel runs in this lesson's folder).
import os
os.environ["LANTERN_CHAPTER"] = "18"
__file__ = os.path.abspath("run.py")

## The scene

The case conference is in nine days. Everything LANTERN has produced so far will be put in front
of Marcus Hale, and Hale has already said what he will ask: which instructions, which model, which
settings, and can you reproduce it.

A demonstration answers none of that. A demonstration is the one run that happened to work. Varma
wants a test suite: every prompt, on every model, against every piece of evidence with a known
right answer, run the same way every time.

## The concept

An **eval** is a test for a prompt. You fix the inputs, you state what a correct output looks
like, and you run the prompt (on one or more models) and score every output automatically. When
you change a prompt or a model, you run the eval again and compare.

promptfoo is an open-source command-line tool for exactly that. It runs locally, talks to Ollama
directly, stores every result, and has a web viewer for reading them.

```mermaid
flowchart LR
    subgraph Config["promptfooconfig.yaml"]
      P[prompts<br/>4 labelled files]
      V[providers<br/>2 local models]
      T[tests<br/>23 cases with vars + asserts]
    end
    P & V & T --> M[Matrix:<br/>each provider x each prompt x each test that names that prompt]
    M --> R[Run every cell<br/>temperature 0, seed 7]
    R --> A[Score every output<br/>with its assertions]
    J[judge qwen3:4b<br/>calibrated on 24 labels] --> A
    A --> O[CLI summary<br/>output/results.json<br/>web viewer]
```

## How it works

![The promptfoo matrix](../../docs/diagrams/18-promptfoo-matrix.svg)

Everything lives in [`evals/`](../../evals/):

```
evals/
  promptfooconfig.yaml      prompts, providers, tests, options
  promptfooconfig.hosted.yaml  the same suite on the hosted model (TIER=hosted)
  providers/
    lantern_provider.py     calls lantern.chat; with a schema, chapter 08's constrained path
  schemas/invoice.json      the invoice schema the extraction suite is constrained to
  judge/
    judge.py                the no-speculation rubric and its judge model (qwen3:4b)
    labels.jsonl            24 answers labelled pass or fail by a person
    calibrate.py            judge against labels: agreement, kappa (make judge-calibrate)
    assert_no_speculation.py  the promptfoo assertion; refuses an uncalibrated judge
  prompts/
    extract-invoice.json    system + user messages, {{invoice}} variable
    classify-v1.json        a terse one-line prompt
    classify-v2.json        definitions + three examples + delimiters
    evidence-rules.json     LANTERN's standing orders as a system prompt
  build_tests.py            generates tests/*.yaml from the case evidence
  tests/
    extraction.yaml         8 invoices, expected totals from the seized ledger
    classification.yaml     12 messages (4 per label) with gold labels
    rules.yaml              3 questions about Petrov's statement
  output/results.json       the last run, every cell
```

**Prompts** are files with a `label`. A chat prompt is a JSON list of messages. `{{invoice}}` and
`{{text}}` are Nunjucks variables filled from each test.

**Providers** are the models. Both use the same settings, so the only difference is the model.
Every prompt goes through `providers/lantern_provider.py`, which calls `lantern.chat`: the
extraction prompt with the invoice schema (chapter 08's schema-constrained path, so the reply is
JSON by construction), the classification and rules prompts as plain chat. Every call is logged
with its prompt version, model, tokens and latency:

```yaml
providers:
  - id: file://providers/lantern_provider.py
    label: qwen3-1.7b
    config: &settings
      model: "qwen3:1.7b"
      temperature: 0
      seed: 7
      max_tokens: 300
      think: false
      prompt_version: v1
      schemas: {extract-invoice: "file://schemas/invoice.json"}
  - id: file://providers/lantern_provider.py
    label: qwen3-0.6b
    config: {<<: *settings, model: "qwen3:0.6b"}
```

An earlier version of this file sent the classification prompts to promptfoo's own `ollama:chat`
provider and only the extraction prompt through `lantern.chat`, with a `prompts:` filter on each
provider. With two providers sharing a label, promptfoo applied the filters wrongly and ran only
the extraction suite: 32 results where 70 were due, and every suite still "green". Count the
results an eval produced, not only how many passed.

**Tests** give each variable a value and list the assertions. `prompts:` limits a test to the
prompts it makes sense for. Without it, promptfoo would send invoices to the classifier.

```yaml
- description: extract NW-0388
  prompts: [extract-invoice]
  vars:
    invoice: file://../case/invoices/NW-0388.txt   # file contents become the variable
    ref: NW-0388
    ledger_total: 62500.0                          # from case/ledger.csv
  assert:
    - type: is-json
      metric: valid_json
    - type: javascript
      metric: invoice_id
      value: JSON.parse(output).invoice_id === context.vars.ref
    - type: javascript
      metric: total_matches_ledger
      value: typeof JSON.parse(output).total === 'number' &&
        Math.abs(JSON.parse(output).total - context.vars.ledger_total) < 0.01
```

**Assertions** used in this suite, from cheapest and most reliable to most expensive:

| Type | What it checks | Used for |
|---|---|---|
| `is-json` | The output parses as JSON | Extraction |
| `javascript` | Any expression over `output` and `context.vars` returns true | Field values, labels |
| `contains` / `icontains` | A substring is present (case sensitive or not) | "23:52", "Not in evidence" |
| `python` | A Python function returns pass or fail with a reason; here, a calibrated model judge | "Does not name anyone as responsible" |

**The matrix.** 2 providers x (8 extraction + 12 classification x 2 prompts + 3 rules) = **70 cells**,
each scored by its own assertions.

## The lesson

The code from `run.py`, one cell per step.

In [ ]:
import json
import os
import re
import subprocess
import sys
import time
from collections import defaultdict

import yaml

from lantern import learn, load_config
from lantern.config import ROOT, tier
from lantern.ui import (
    Table,
    chapter,
    console,
    evidence,
    finding,
    note,
    save_shot,
    step,
    your_turn,
)

chapter("18", "Proven, not demonstrated", """
The case conference is in nine days. Everything LANTERN has produced so far will be put in front
of Marcus Hale, and Hale has already said what he will ask: which instructions, which model, which
settings, and can you reproduce it. A demonstration answers none of that. A demonstration is the
one run that happened to work. Varma wants a test suite: every prompt, on every model, against
every piece of evidence with a known right answer, run the same way every time.
""")

EVALS = ROOT / "evals"
PROMPTFOO = ["npx", "--yes", "promptfoo@0.124.1"]
# The hosted tier runs the same tests through lantern.chat on the hosted model, into its own
# output files, so a hosted run never overwrites the committed local results.
HOSTED = tier() == "hosted"
CONFIG = "promptfooconfig.hosted.yaml" if HOSTED else "promptfooconfig.yaml"
SUFFIX = ".hosted" if HOSTED else ""

### 1. The suite is a config file

In [ ]:
step("1. The suite is a config file")
config = (EVALS / CONFIG).read_text(encoding="utf-8")
evidence(f"evals/{CONFIG}", "\n".join(
    line for line in config.splitlines() if line.strip() and not line.lstrip().startswith("#")))
note("prompts x providers x tests: each test names the prompts it applies to.")

### 2. Tests are generated from the evidence

In [ ]:
step("2. Tests are generated from the evidence")
build = subprocess.run([sys.executable, str(EVALS / "build_tests.py")], capture_output=True,
                       text=True, check=True)
note(build.stdout.strip())
sample = (EVALS / "tests" / "extraction.yaml").read_text(encoding="utf-8").split("- description")
evidence("evals/tests/extraction.yaml (first test)", "- description" + sample[1])
note("Expected totals come from the seized ledger, case/ledger.csv: the record we must verify.")

tests_now = sum(len(yaml.safe_load((EVALS / "tests" / f).read_text(encoding="utf-8")))
                for f in ("extraction.yaml", "classification.yaml", "rules.yaml"))

### 3. promptfoo eval

In [ ]:
step("3. promptfoo eval")
summary_file = EVALS / "output" / f"eval-summary{SUFFIX}.txt"
results_file = EVALS / "output" / f"results{SUFFIX}.json"
reused = "--reuse" in sys.argv and summary_file.exists() and results_file.exists()
minutes = None
p_pass = learn.predict_number("How many cells will pass? (The unchanged suite has 70.)",
                              unit="cells")
if HOSTED and "--reuse" in sys.argv and not reused:
    # No hosted eval has been recorded: say so, and read the local run rather than invent one.
    note("--reuse on the hosted tier: no hosted eval has been recorded (not run). Showing the "
         "local run instead; `make lesson N=18 TIER=hosted` with a key records one.")
    summary_file = EVALS / "output" / "eval-summary.txt"
    results_file = EVALS / "output" / "results.json"
    reused = True
if reused:
    note("--reuse: showing the last run instead of running the eval again.")
    header, *summary = summary_file.read_text(encoding="utf-8").splitlines()
else:
    env = {**os.environ, "OLLAMA_BASE_URL": load_config()["ollama"]["host"],
           "PROMPTFOO_DISABLE_TELEMETRY": "1", "PROMPTFOO_DISABLE_UPDATE": "1"}
    # The Python provider and the judge need lantern: run them with this interpreter.
    env.setdefault("PROMPTFOO_PYTHON", sys.executable)
    note("Running 70 cells one at a time on CPU, plus the judge. This takes several minutes.")
    start = time.perf_counter()
    run = subprocess.run([*PROMPTFOO, "eval", "--no-cache", "--no-table", "--no-progress-bar",
                          "-c", CONFIG, "--output", f"output/results{SUFFIX}.json"],
                         cwd=EVALS, env=env, capture_output=True, text=True)
    minutes = (time.perf_counter() - start) / 60
    summary = [line for line in re.sub(r"\x1b\[[0-9;]*m", "", run.stdout).splitlines()
               if line.strip()][-18:]
    header = f"promptfoo eval (exit code {run.returncode}, {minutes:.1f} min)"
    summary_file.write_text("\n".join([header, *summary]) + "\n", encoding="utf-8")
    # results.json is committed: take this machine's home directory out of it.
    subprocess.run([sys.executable, str(ROOT / "scripts" / "clean_eval_output.py")],
                   capture_output=True, text=True)
evidence(header, "\n".join(summary))
note("Exit code 100 means the run finished and some tests failed. That is a result, not a crash.")
data = json.loads(results_file.read_text(encoding="utf-8"))
rows = data["results"]["results"]
passed_cells = sum(bool(r["success"]) for r in rows)
learn.reveal(p_pass, actual=passed_cells, tolerance=0.2,
             why=f"{passed_cells} of {len(rows)} passed. A pass rate is a property of the prompts, "
                 "the models and the tests together: the same prompts against harder tests would "
                 "score lower. Step 4 shows where the failures sit.")

### 4. The matrix: pass rate per model and prompt

In [ ]:
step("4. The matrix: pass rate per model and prompt")
cells = defaultdict(lambda: [0, 0])
for r in rows:
    key = (r["provider"].get("label") or r["provider"]["id"], r["prompt"]["label"])
    cells[key][0] += bool(r["success"])
    cells[key][1] += 1
providers = sorted({k[0] for k in cells})
prompts = sorted({k[1] for k in cells})
p_low = learn.predict("Which prompt will pass the smallest share of its tests, over both models?",
                      prompts)
table = Table(title="Tests passed (passed / run)")
table.add_column("prompt")
for p in providers:
    table.add_column(p, justify="right")
for prompt in prompts:
    row = [prompt]
    for p in providers:
        ok, n = cells[(p, prompt)]
        colour = "green" if ok == n else "yellow" if ok >= n / 2 else "red"
        row.append(f"[{colour}]{ok}/{n}[/{colour}]")
    table.add_row(*row)
console.print(table)
by_prompt = {pr: [sum(cells[(p, pr)][i] for p in providers) for i in (0, 1)] for pr in prompts}
lowest = min(prompts, key=lambda pr: by_prompt[pr][0] / max(by_prompt[pr][1], 1))
learn.reveal(p_low, answer=prompts.index(lowest),
             why=f"{lowest} passed {by_prompt[lowest][0]} of {by_prompt[lowest][1]}. Same models, "
                 "same settings: when one prompt fails where another passes on the same tests, "
                 "the prompt is the cause. Compare classify-v1 with classify-v2.")
stats = data["results"]["stats"]
note(f"Overall: {stats['successes']} passed, {stats['failures']} failed, "
     f"{stats.get('errors', 0)} errors.")
note("Open the full results with `make eval-view` (http://localhost:15500).")

### 5. Who grades the grader: the calibrated judge

In [ ]:
step("5. Who grades the grader: the calibrated judge")
cal_path = EVALS / "judge" / "calibration.json"
cal = json.loads(cal_path.read_text(encoding="utf-8")) if cal_path.exists() else None
if cal:
    c = cal["confusion"]
    table = Table(title=f"{cal['judge_model']} against {cal['n']} answers a person labelled "
                        f"({cal['date']}, {cal['tier']}, {cal['machine']})")
    table.add_column("")
    table.add_column("judge: pass", justify="right")
    table.add_column("judge: fail", justify="right")
    table.add_row("label: pass", str(c["label_pass_judge_pass"]), str(c["label_pass_judge_fail"]))
    table.add_row("label: fail", str(c["label_fail_judge_pass"]), str(c["label_fail_judge_fail"]))
    console.print(table)
    usable = cal["kappa"] >= cal["min_kappa"]
    note(f"Agreement {cal['agreement']:.2f}, true-pass rate {cal['true_pass_rate']:.2f}, "
         f"true-fail rate {cal['true_fail_rate']:.2f}, Cohen's kappa {cal['kappa']:.2f} "
         f"({cal['min_kappa']} needed): "
         + ("[green]usable[/green]." if usable else "[red]NOT usable: it fails closed[/red]."))
else:
    usable = False
    note("[red]No evals/judge/calibration.json: the judge has not been checked against the "
         "labels, so every no_speculation cell fails closed. Run `make judge-calibrate`.[/red]")
verdicts = []
for r in rows:
    for comp in (r.get("gradingResult") or {}).get("componentResults") or []:
        if (comp.get("assertion") or {}).get("metric") == "no_speculation":
            verdicts.append((r["provider"].get("label") or r["provider"]["id"],
                             bool(comp.get("pass")), comp.get("reason") or ""))
judged = [v for v in verdicts if v[2].startswith("judge ")]
refused_self = sum("its own model" in v[2] for v in verdicts)
p_judge = learn.predict(
    "Asked 'Who pushed Mr Reyes down the stairs?', how many of the models under test will the "
    "judge pass?", ["all of them", "some of them", "none of them"])
for prov, ok, reason in verdicts:
    note(f"no_speculation, {prov}: {'pass' if ok else '[red]fail[/red]'} ({reason[:110]})")
judge_passed = sum(v[1] for v in verdicts)
learn.reveal(p_judge, answer=0 if verdicts and judge_passed == len(verdicts) else
             2 if judge_passed == 0 else 1,
             why="A judge is a model grading a model. Its verdict is only worth what its "
                 "agreement with a person is worth, which is why it is calibrated against labels "
                 "first and is never the model under test: a model grading itself shares its own "
                 "blind spots.")

your_turn(
    "18",
    check=[
        "In the matrix, compare classify-v1 with classify-v2 on the same model. Then open "
        "evals/prompts/classify-v1.json and evals/prompts/classify-v2.json. Which lines could "
        "explain the gap?",
        "Find the extract-invoice cells that failed. Run make eval-view, filter to failures, and "
        "compare the two models' outputs for that test.",
        "Open evals/tests/rules.yaml. Which assertion is graded by a model instead of by code, "
        "and would you trust it on its own?",
    ],
    exercises=[
        "In evals/promptfooconfig.yaml, under `providers:`, add a third lantern provider with "
        "`label: qwen3-4b` and `config: {<<: *settings, model: \"qwen3:4b\"}`. Compare the "
        "extra minutes with the "
        "extra passes.",
        "Add finance to the label list in evals/prompts/classify-v1.json and "
        "evals/prompts/classify-v2.json. In `classification_tests()` in evals/build_tests.py, "
        "add a test whose `label` var is finance. What does v1 do with it?",
        "In `extraction_tests()` in evals/build_tests.py, replace the `total_matches_ledger` "
        "javascript assert with an llm-rubric whose value is 'The total matches "
        "{{ledger_total}}.' Compare the run time and the grader's mistakes.",
        "In evals/tests/extraction.yaml, delete `prompts:` and `- extract-invoice` from the "
        "extract NW-0402 test. Run make eval (this lesson rebuilds that file), then make "
        "eval-view and read what the classify prompts made of it.",
    ],
)

learn.record(
    cells=len(rows),
    passed=passed_cells,
    failed=len(rows) - passed_cells,
    pass_rate=round(passed_cells / max(len(rows), 1), 3),
    providers=len(providers),
    tests=tests_now,
    extract_passed=by_prompt.get("extract-invoice", [0, 0])[0],
    classify_v1_passed=by_prompt.get("classify-v1", [0, 0])[0],
    classify_v2_passed=by_prompt.get("classify-v2", [0, 0])[0],
    rules_passed=by_prompt.get("evidence-rules", [0, 0])[0],
    judge_kappa=round(cal["kappa"], 3) if cal else None,
    judge_agreement=round(cal["agreement"], 3) if cal else None,
    judge_usable=usable,
    judge_graded=len(judged),
    judge_refused_self=refused_self,
    minutes=round(minutes, 1) if minutes is not None else None,
    reused=reused,
)

finding("""
The suite runs end to end and every result is stored with its prompt, model, settings and
output. Hale can have all of it. The rules hold on both models. Extraction is nearly perfect. The
terse classifier fails every test it is given, and the better one still misses the quietest
threats. And one extraction test fails on both models in exactly the same way.

Varma looks at the matrix for a long time. "Before you fix anything, find out what is failing.
A failing test is evidence too."
""")
save_shot("18-evals-with-promptfoo")

## The nuances

**1. Deterministic assertions first.** `is-json`, `javascript` and `contains` are free, fast and
never disagree with themselves. Use them for everything they can express. Reach for `llm-rubric`
only when correctness genuinely needs judgement, and remember the grader is itself a model that
can be wrong. This suite uses one judge, qwen3:4b, on one rubric, and nuance 11 says why it may.

**2. Expected values have a source, and the source can be wrong.** The extraction tests take their
expected totals from the seized ledger, because that is the record the EOW must verify. An eval is
only as trustworthy as its reference data. Chapter 19 is about what happens when the reference is
the thing that is wrong.

**3. What you do not assert, you do not know.** The 1.7B model put "Halcyon Maritime Logistics"
(the company being billed) in the `vendor` field of several invoices. Every one of those tests
passed, because no assertion checks `vendor`. A green cell means "the checks you wrote passed", not
"the output is right".

**4. Name your metrics.** `metric: total_matches_ledger` turns a pile of pass/fail results into a
score per property. In the viewer you can see that a model gets JSON right 100% of the time but
totals right only 80%, which tells you where to work.

**5. Constrain, do not transform.** An earlier version of this suite stripped code fences with a
regex `transform`, because small models add them despite the instruction. That hid a format
failure instead of removing it. The extraction suite now runs through `lantern.chat` with the
invoice schema: Ollama turns the schema into a grammar (a hosted provider uses its native
structured output), so the reply cannot have fences. `is-json` still runs, because "by
construction" is a claim worth checking, and the total check now asks for a number, so a string
that `Number()` would quietly coerce does not pass.

**6. Turn the cache off while developing prompts.** promptfoo caches responses by default.
`--no-cache` (used by `make eval`) guarantees every cell really ran. With temperature 0 and a seed,
a re-run should match, and chapter 06 explains why it sometimes does not.

**7. Concurrency is a resource decision.** `maxConcurrency: 1` sends one request at a time. On a GPU
server you would raise it. Here, two models alternating under heavy parallel load would mean
constant reloading.

**8. Generate tests from data, do not type them.** `build_tests.py` reads the case files. When new
evidence arrives, regenerate and rerun. Hand-typed expected values drift from their source.

**9. Telemetry.** promptfoo sends anonymous usage telemetry by default. The lesson sets
`PROMPTFOO_DISABLE_TELEMETRY=1` and `PROMPTFOO_DISABLE_UPDATE=1`. Do the same for anything sensitive.

**10. Exit codes are part of the contract.** `promptfoo eval` exits 100 when tests fail. In CI that
fails the build, which is what you want: a prompt change that breaks a test should not merge.

**11. A judge is calibrated against labels, and never grades its own model.** The no-speculation
rubric used to be an `llm-rubric` graded by qwen3:1.7b, one of the two models under test. That is
two mistakes. A model grading its own answers shares their blind spots: a hint at a suspect it
did not notice writing, it will not notice reading. And nobody had measured whether the grader
agreed with a person at all. Now the rubric is a `python` assertion
([`judge/assert_no_speculation.py`](../../evals/judge/assert_no_speculation.py)) whose judge is
qwen3:4b, a model not under test. Before it may grade, `make judge-calibrate` runs it on 24
answers a person labelled (12 pass, 12 fail, several that break the rule only by hinting, "the
person with the code") and reports agreement, the true-pass and true-fail rates, and Cohen's
kappa: agreement beyond chance. A judge that passes everything agrees on half a balanced set and
has a kappa of 0. The assertion fails closed: no `calibration.json`, a calibration made for
another model, or a kappa under 0.6, and every `no_speculation` cell fails with the reason. It
also refuses to grade a provider whose model is the judge's own. The calibration numbers for
this suite, from `evals/judge/calibration.json` (23 November 2026, local tier): 24 of 24 labels
agreed, kappa 1.00. A perfect score on 24 answers a person wrote is a warning as much as a pass:
it says the labels may be too easy to tell a good judge from a lucky one. Add harder ones (an
answer that blames a role without a name, a hedge buried in a long reply) before you trust it on
more than one rubric.

## On a hosted model

```bash
make lesson N=18 TIER=hosted            # needs ANTHROPIC_API_KEY; RECORD=1 saves the run
make lesson N=18 TIER=hosted REPLAY=1   # once a run is recorded: no key, no network
```

On the hosted tier the lesson runs `evals/promptfooconfig.hosted.yaml`: the same prompts, tests
and assertions, with every prompt sent through `providers/lantern_provider.py` on the hosted tier.
The extraction prompt uses the provider's **native structured output** with the invoice schema;
the other prompts are plain chat. Claude takes no `temperature` and no `seed`: `lantern.cloud`
does not send them and reports them as not applied, on the console and in each cell's metadata,
so the hosted run is not reproducible in the way chapter 06 made the local one. The judge stays
the calibrated local qwen3:4b (the assertion pins it to the tier it was calibrated on), so a
hosted model is never graded by an uncalibrated judge. Hosted results go to
`evals/output/results.hosted.json` and never overwrite the local ones. With `--reuse` and no
recorded hosted run, the lesson says so and shows the local run.

| Measure | Local: qwen3-1.7b / qwen3-0.6b, Ollama, Apple M1 Pro 16 GB | Hosted: claude-haiku-5-5 |
|---|---|---|
| extract-invoice passed | 7/8 / 7/8 | not run |
| classify-v1 passed | 0/12 / 0/12 | not run |
| classify-v2 passed | 8/12 / 6/12 | not run |
| evidence-rules passed | 3/3 / 2/3 | not run |
| cells passed, whole suite | 33 of 70 | not run |

Live hosted numbers are not run: this lab has no hosted key, and no number here is estimated.

What the hosted run will test: whether classify-v1's paragraph answers are a small-model effect (a
larger model may follow "which label" more literally, or explain itself just the same), and
whether NW-0419 fails on Claude exactly as it did on both local models. It should: the invoice
says 94,000.00 and the ledger 9,400.00, whatever reads it.

## Try this

1. Add `qwen3:4b` as a third provider and rerun. Is the extra time worth the extra passes?
2. Add a fourth label to the classification prompts (`finance`) and tests for it. What happens to
   v1, which has no definitions?
3. Replace the `javascript` total check with an `llm-rubric` ("the total matches {{ledger_total}}")
   and compare the time taken and how often the grader is wrong.
4. Delete `prompts:` from one extraction test and run again. Read what the classifier does with an
   invoice.

## Master it

After this chapter you can describe an eval as prompts, providers and tests, write its
assertions as code, and decide whether a model judge has earned a place in it.

**Predict, then see.** `make lesson N=18` stops three times to ask: how many of the 70 cells will
pass, which prompt will pass the smallest share of its tests, and how many of the models under
test the judge will pass on "Who pushed Mr Reyes down the stairs?". Each answer is checked
against your own run (or the last one, with `--reuse`), not a fixed key. Add `ARGS=--no-predict`
to watch without stopping.

**Build it.** Write the suite's graders in Python in
[`practice/exercise.py`](practice/exercise.py): the three extraction metrics, the one-word label
check, and a per-metric score.

```bash
make exercise N=18          # 8 offline tests, no model and no promptfoo needed
make hint N=18              # one hint at a time
make solution N=18          # the reference, once you have tried
```

**Check yourself.** `make quiz N=18`: seven questions, each with the reason behind the answer.

**Break it.** `make challenge N=18` checks your latest run against two targets: qwen3:4b added as
a model under test, so the judge has to refuse to grade its own model; and the one-line
classifier made to pass at least 12 of its 24 cells.

`make progress` shows every chapter.

The cells below run the same commands inside this notebook.

### Check yourself

In [ ]:
%run ../../scripts/learn.py quiz 18

### Build it

Open [`practice/exercise.py`](practice/exercise.py) beside this notebook, fill in the TODOs, save, and run the grader. Hints come one at a time.

In [ ]:
%run ../../scripts/learn.py exercise 18

In [ ]:
%run ../../scripts/learn.py hint 18

### Break it

The challenges are checked against your latest run of the lesson cells above (or of `make lesson`). Change the code, rerun the lesson, then check.

In [ ]:
%run ../../scripts/learn.py challenge 18

In [ ]:
%run ../../scripts/learn.py progress

## What the evidence now shows

The suite runs end to end, and every result is stored with its prompt, model, settings and output.
Hale can have all of it. Most cells pass. Some do not, and one test fails in a way you did not
expect.

Varma looks at the matrix for a long time. "Before you fix anything, find out what is failing. A
failing test is evidence too."

**Next:** [Chapter 19: The test that would not pass](../19-reading-eval-results/README.md)